### HGT with Edge Awareness (Weights), and Positional Awareness (Laplacian PE)

In [ ]:
import torch
import numpy as np
import math
import networkx as nx
from torch_geometric.data import HeteroData
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import KFold
import sys
from pathlib import Path
from torch_geometric.utils import to_undirected, softmax
from scipy import sparse

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

print("python:", sys.executable)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

data_dir = Path("../data")
tcr_features = np.load(data_dir / 'TCR_tcrbert_embeddings.npy')
epitope_features = np.load(data_dir / 'epitope_esm2_embeddings.npy')
tcr_similarity = np.load(data_dir / 'TCR_similarity_combined.npy')
epitope_similarity = np.load(data_dir / 'Epitope_similarity_combined.npy')
associations = np.load(data_dir / 'association.npy')
print("Data loaded successfully.")
print("associations shape:", associations.shape)


In [ ]:
num_tcr = tcr_features.shape[0]
num_epitope = epitope_features.shape[0]

print("Building Graph...")
G = nx.Graph()
for i in range(num_tcr):
    G.add_node(f'TCR_{i}', type='TCR', x=tcr_features[i])
for i in range(num_epitope):
    G.add_node(f'Epitope_{i}', type='Epitope', x=epitope_features[i])

k_tcr = 20
for i, row in enumerate(tcr_similarity):
    idx = np.argpartition(row, -k_tcr)[-k_tcr:]
    idx = [j for j in idx if j != i]
    for j in idx:
        G.add_edge(f'TCR_{i}', f'TCR_{j}', edge_type='TCR-TCR', weight=float(tcr_similarity[i, j]))

k_epi = 15
for i, row in enumerate(epitope_similarity):
    idx = np.argpartition(row, -k_epi)[-k_epi:]
    idx = [j for j in idx if j != i]
    for j in idx:
        G.add_edge(f'Epitope_{i}', f'Epitope_{j}', edge_type='Epitope-Epitope', weight=float(epitope_similarity[i, j]))

print("Calculating node degrees...")
def get_node_degrees(G, num_nodes, prefix, max_degree=64):
    degrees = []
    for i in range(num_nodes):
        node_name = f'{prefix}_{i}'
        d = G.degree[node_name] if node_name in G else 0
        d = min(d, max_degree) 
        degrees.append(d)
    return torch.tensor(degrees, dtype=torch.long)

tcr_degrees = get_node_degrees(G, num_tcr, 'TCR', max_degree=64)
epi_degrees = get_node_degrees(G, num_epitope, 'Epitope', max_degree=64)

print("Computing Laplacian positional encodings...")
def compute_laplacian_pe(G, num_nodes, prefix, k=8):
    nodes = [f'{prefix}_{i}' for i in range(num_nodes)]
    sub_G = G.subgraph(nodes)
    
    node_map = {n: i for i, n in enumerate(nodes)}
    
    edges = []
    for u, v in sub_G.edges():
        if u in node_map and v in node_map:
            edges.append((node_map[u], node_map[v]))
            edges.append((node_map[v], node_map[u]))
            
    if not edges:
        return torch.zeros((num_nodes, k))

    row, col = zip(*edges)
    data = np.ones(len(row))
    adj = sparse.coo_matrix((data, (row, col)), shape=(num_nodes, num_nodes))
    
    degrees = np.array(adj.sum(1)).flatten()
    d_inv_sqrt = np.power(degrees, -0.5, where=degrees>0)
    d_inv_sqrt[degrees==0] = 0
    
    D_inv_sqrt = sparse.diags(d_inv_sqrt)
    L = sparse.eye(num_nodes) - D_inv_sqrt @ adj @ D_inv_sqrt
    
    try:
        evals, evecs = sparse.linalg.eigsh(L, k=k+1, which='SM', tol=1e-2)
        pe = evecs[:, 1:k+1] 
        
        if pe.shape[1] < k:
            pad = np.zeros((pe.shape[0], k - pe.shape[1]))
            pe = np.hstack([pe, pad])
            
    except Exception as e:
        print(f"LPE Computation failed for {prefix}: {e}. Using zeros.")
        pe = np.zeros((num_nodes, k))
        
    return torch.from_numpy(pe).float()

pe_dim = 8
tcr_pe = compute_laplacian_pe(G, num_tcr, 'TCR', k=pe_dim)
epi_pe = compute_laplacian_pe(G, num_epitope, 'Epitope', k=pe_dim)

data = HeteroData()
data['TCR'].x = torch.tensor(tcr_features, dtype=torch.float)
data['Epitope'].x = torch.tensor(epitope_features, dtype=torch.float)

data['TCR'].degree = tcr_degrees
data['Epitope'].degree = epi_degrees
data['TCR'].pe = tcr_pe
data['Epitope'].pe = epi_pe

def extract_edges_and_weights(G, edge_type):
    edges = []
    weights = []
    for u, v, d in G.edges(data=True):
        if d.get('edge_type') == edge_type:
            u_idx = int(u.split('_')[1])
            v_idx = int(v.split('_')[1])
            edges.append((u_idx, v_idx))
            weights.append(d['weight'])
            edges.append((v_idx, u_idx))
            weights.append(d['weight'])
    return torch.tensor(edges).t(), torch.tensor(weights, dtype=torch.float)

tcr_edge_index, tcr_weights = extract_edges_and_weights(G, 'TCR-TCR')
data['TCR', 'connected_to', 'TCR'].edge_index = tcr_edge_index
data['TCR', 'connected_to', 'TCR'].edge_attr = tcr_weights

epi_edge_index, epi_weights = extract_edges_and_weights(G, 'Epitope-Epitope')
data['Epitope', 'connected_to', 'Epitope'].edge_index = epi_edge_index
data['Epitope', 'connected_to', 'Epitope'].edge_attr = epi_weights

pos_i, pos_j = np.where(associations == 1)
edge_index = torch.tensor([pos_i, pos_j], dtype=torch.long)
data['TCR', 'associated_with', 'Epitope'].edge_index = edge_index

class EdgeAwareGraphTransformerBlock(torch.nn.Module):
    def __init__(self, channels, heads=4, dropout=0.3):
        super().__init__()
        self.heads = heads
        self.head_dim = channels // heads
        self.scale = math.sqrt(self.head_dim)
        
        self.q_proj = torch.nn.Linear(channels, channels)
        self.k_proj = torch.nn.Linear(channels, channels)
        self.v_proj = torch.nn.Linear(channels, channels)
        
        self.edge_encoder = torch.nn.Linear(1, heads)
        
        self.out_proj = torch.nn.Linear(channels, channels)
        self.norm1 = torch.nn.LayerNorm(channels)
        self.dropout1 = torch.nn.Dropout(dropout)
        
        self.ffn = torch.nn.Sequential(
            torch.nn.Linear(channels, channels * 2),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(channels * 2, channels)
        )
        self.norm2 = torch.nn.LayerNorm(channels)
        self.dropout2 = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index, edge_attr=None):
        if edge_index.numel() == 0: return x

        src, dst = edge_index
        q = self.q_proj(x).view(-1, self.heads, self.head_dim)
        k = self.k_proj(x).view(-1, self.heads, self.head_dim)
        v = self.v_proj(x).view(-1, self.heads, self.head_dim)
        
        attn_scores = (q[dst] * k[src]).sum(dim=-1) / self.scale
        
        if edge_attr is not None:
            edge_bias = self.edge_encoder(edge_attr.view(-1, 1))
            attn_scores = attn_scores + edge_bias
            
        alpha = softmax(attn_scores, dst, num_nodes=x.size(0))
        
        messages = v[src] * alpha.unsqueeze(-1)
        attn_out = torch.zeros_like(q)
        attn_out.index_add_(0, dst, messages)
        attn_out = attn_out.view(-1, self.heads * self.head_dim)
        attn_out = self.out_proj(attn_out)
        
        x = self.norm1(x + self.dropout1(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out))
        return x

class CrossAttentionBlock(torch.nn.Module):
    def __init__(self, embed_dim, heads=4, dropout=0.3):
        super().__init__()
        self.heads = heads
        self.head_dim = embed_dim // heads
        self.scale = math.sqrt(self.head_dim)
        
        self.q_proj = torch.nn.Linear(embed_dim, embed_dim)
        self.k_proj = torch.nn.Linear(embed_dim, embed_dim)
        self.v_proj = torch.nn.Linear(embed_dim, embed_dim)
        self.out_proj = torch.nn.Linear(embed_dim, embed_dim)
        
        self.norm1 = torch.nn.LayerNorm(embed_dim)
        self.dropout1 = torch.nn.Dropout(dropout)
        
        self.ffn = torch.nn.Sequential(
            torch.nn.Linear(embed_dim, embed_dim * 2),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(embed_dim * 2, embed_dim),
        )
        self.norm2 = torch.nn.LayerNorm(embed_dim)
        self.dropout2 = torch.nn.Dropout(dropout)

    def forward(self, q_x, kv_x, edge_index):
        if edge_index.numel() == 0: return q_x
        src, dst = edge_index
        q = self.q_proj(q_x).view(-1, self.heads, self.head_dim)
        k = self.k_proj(kv_x).view(-1, self.heads, self.head_dim)
        v = self.v_proj(kv_x).view(-1, self.heads, self.head_dim)

        attn_scores = (q[src] * k[dst]).sum(dim=-1) / self.scale
        alpha = softmax(attn_scores, src)
        alpha = self.dropout1(alpha)

        messages = v[dst] * alpha.unsqueeze(-1)
        out = torch.zeros_like(q)
        out.index_add_(0, src, messages)
        out = out.view(-1, self.heads * self.head_dim)
        
        attn_out = self.out_proj(out)
        q_x = self.norm1(q_x + self.dropout1(attn_out))
        ffn_out = self.ffn(q_x)
        q_x = self.norm2(q_x + self.dropout2(ffn_out))
        return q_x

class FullAwareTransformer(torch.nn.Module):
    def __init__(self, in_channels_tcr, in_channels_epitope, hidden_channels, 
                 heads=4, dropout=0.3, max_degree=64, pe_dim=8, num_layers=2):
        super().__init__()
        
        self.tcr_encoder = torch.nn.Sequential(
            torch.nn.Linear(in_channels_tcr, hidden_channels),
            torch.nn.LayerNorm(hidden_channels),
            torch.nn.ReLU(),
        )
        self.epitope_encoder = torch.nn.Sequential(
            torch.nn.Linear(in_channels_epitope, hidden_channels),
            torch.nn.LayerNorm(hidden_channels),
            torch.nn.ReLU(),
        )

        self.pe_lin = torch.nn.Linear(pe_dim, hidden_channels)
        self.pe_norm = torch.nn.LayerNorm(pe_dim)

        self.num_layers = num_layers
        self.tcr_gt_blocks = torch.nn.ModuleList([
            EdgeAwareGraphTransformerBlock(hidden_channels, heads=heads, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.epi_gt_blocks = torch.nn.ModuleList([
            EdgeAwareGraphTransformerBlock(hidden_channels, heads=heads, dropout=dropout)
            for _ in range(num_layers)
        ])

        self.tcr_cross = CrossAttentionBlock(hidden_channels, heads=heads, dropout=dropout)
        self.epitope_cross = CrossAttentionBlock(hidden_channels, heads=heads, dropout=dropout)
        
        self.feature_dropout = torch.nn.Dropout(dropout)

        self.edge_classifier = torch.nn.Sequential(
            torch.nn.Linear(hidden_channels * 2, hidden_channels),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden_channels, 1),
        )

    def forward(self, data, edge_index):
        tcr_x = self.tcr_encoder(data['TCR'].x)
        epitope_x = self.epitope_encoder(data['Epitope'].x)

        if hasattr(data['TCR'], 'pe'):
            tcr_pe = self.pe_lin(self.pe_norm(data['TCR'].pe))
            tcr_x = tcr_x + tcr_pe
        if hasattr(data['Epitope'], 'pe'):
            epi_pe = self.pe_lin(self.pe_norm(data['Epitope'].pe))
            epitope_x = epitope_x + epi_pe

        tcr_edge_index = data['TCR', 'connected_to', 'TCR'].edge_index
        tcr_edge_attr = data['TCR', 'connected_to', 'TCR'].edge_attr.to(edge_index.device)
        
        if tcr_edge_index.numel() > 0:
            for layer in self.tcr_gt_blocks:
                tcr_x = layer(tcr_x, tcr_edge_index, tcr_edge_attr)

        epi_edge_index = data['Epitope', 'connected_to', 'Epitope'].edge_index
        epi_edge_attr = data['Epitope', 'connected_to', 'Epitope'].edge_attr.to(edge_index.device)
        
        if epi_edge_index.numel() > 0:
            for layer in self.epi_gt_blocks:
                epitope_x = layer(epitope_x, epi_edge_index, epi_edge_attr)

        assoc_edge_index = data['TCR', 'associated_with', 'Epitope'].edge_index.to(edge_index.device)
        tcr_x = self.tcr_cross(tcr_x, epitope_x, assoc_edge_index)
        epitope_x = self.epitope_cross(epitope_x, tcr_x, assoc_edge_index.flip(0).contiguous())

        src, dst = edge_index
        pair_features = torch.cat([tcr_x[src], epitope_x[dst]], dim=-1)
        pair_features = self.feature_dropout(pair_features)

        return self.edge_classifier(pair_features).squeeze(-1)

def sample_negative_edges(num_tcr, num_epitope, associations, num_samples, rng=None):
    pos_rows, pos_cols = np.where(associations == 1)
    pos_edges = set(zip(pos_rows, pos_cols))
    if rng is None:
        rng = np.random
    neg_edges = []
    while len(neg_edges) < num_samples:
        u = rng.randint(0, num_tcr, size=num_samples)
        v = rng.randint(0, num_epitope, size=num_samples)
        mask = [((r, c) not in pos_edges) for r, c in zip(u, v)]
        u, v = u[mask], v[mask]
        neg_edges.extend(list(zip(u, v)))
        if len(neg_edges) > num_samples:
            neg_edges = neg_edges[:num_samples]
    return torch.tensor(neg_edges, dtype=torch.long).t()

data = data.to(device)
num_pos_edges = edge_index.size(1)

k_folds   = 10
n_runs    = 5
run_seeds = [42, 123, 456, 789, 1234]

all_run_auc_scores = []
all_fold_probs     = []
all_fold_labels    = []
cv_split_runs      = []

print(f"Starting {n_runs} × {k_folds}-Fold CV with Full Awareness (Edge + Degree + LPE)...")

for run_idx, seed in enumerate(run_seeds):
    print(f"\n{'='*60}")
    print(f"Run {run_idx + 1}/{n_runs}  (seed={seed})")
    print(f"{'='*60}")

    rng = np.random.RandomState(seed)
    negative_edge_index = sample_negative_edges(num_tcr, num_epitope, associations, num_pos_edges, rng=rng)
    num_neg_edges = negative_edge_index.size(1)

    kf_pos = KFold(n_splits=k_folds, shuffle=True, random_state=seed)
    kf_neg = KFold(n_splits=k_folds, shuffle=True, random_state=seed)

    pos_splits = list(kf_pos.split(np.arange(num_pos_edges)))
    neg_splits = list(kf_neg.split(np.arange(num_neg_edges)))

    run_plan = {
        "run_idx": int(run_idx),
        "seed": int(seed),
        "negative_edge_index": negative_edge_index.cpu().numpy(),
        "folds": []
    }
    for (train_pos_idx, val_pos_idx), (train_neg_idx, val_neg_idx) in zip(pos_splits, neg_splits):
        run_plan["folds"].append({
            "train_pos_idx": np.asarray(train_pos_idx, dtype=np.int64),
            "val_pos_idx": np.asarray(val_pos_idx, dtype=np.int64),
            "train_neg_idx": np.asarray(train_neg_idx, dtype=np.int64),
            "val_neg_idx": np.asarray(val_neg_idx, dtype=np.int64),
        })
    cv_split_runs.append(run_plan)

    run_auc_scores = []

    for fold, ((train_pos_idx, val_pos_idx), (train_neg_idx, val_neg_idx)) in enumerate(zip(pos_splits, neg_splits)):

        print(f"\n  Fold {fold+1}/{k_folds}")

        train_pos_edge = edge_index[:, train_pos_idx].to(device)
        val_pos_edge   = edge_index[:, val_pos_idx].to(device)
        train_neg_edge = negative_edge_index[:, train_neg_idx].to(device)
        val_neg_edge   = negative_edge_index[:, val_neg_idx].to(device)

        train_edges  = torch.cat([train_pos_edge, train_neg_edge], dim=1)
        train_target = torch.cat([torch.ones(train_pos_edge.size(1)),
                                   torch.zeros(train_neg_edge.size(1))], dim=0).to(device)
        val_edges    = torch.cat([val_pos_edge, val_neg_edge], dim=1)
        val_target   = torch.cat([torch.ones(val_pos_edge.size(1)),
                                   torch.zeros(val_neg_edge.size(1))], dim=0).cpu().numpy()

        fold_data = HeteroData()
        fold_data['TCR'].x       = data['TCR'].x
        fold_data['Epitope'].x   = data['Epitope'].x
        fold_data['TCR'].pe      = data['TCR'].pe.to(device)
        fold_data['Epitope'].pe  = data['Epitope'].pe.to(device)
        fold_data['TCR', 'connected_to', 'TCR'].edge_index = data['TCR', 'connected_to', 'TCR'].edge_index
        fold_data['TCR', 'connected_to', 'TCR'].edge_attr  = data['TCR', 'connected_to', 'TCR'].edge_attr
        fold_data['Epitope', 'connected_to', 'Epitope'].edge_index = data['Epitope', 'connected_to', 'Epitope'].edge_index
        fold_data['Epitope', 'connected_to', 'Epitope'].edge_attr  = data['Epitope', 'connected_to', 'Epitope'].edge_attr
        fold_data['TCR', 'associated_with', 'Epitope'].edge_index = train_pos_edge

        model = FullAwareTransformer(
            tcr_features.shape[1],
            epitope_features.shape[1],
            hidden_channels=256,
            heads=2,
            dropout=0.3,
            pe_dim=8,
            num_layers=2
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)

        best_val_auc   = -1.0
        best_val_aupr  = -1.0
        best_val_probs = None
        patience       = 0
        best_epoch     = 0

        for epoch in range(150):
            model.train()
            optimizer.zero_grad()
            logits = model(fold_data, train_edges)
            loss   = F.binary_cross_entropy_with_logits(logits, train_target)
            loss.backward()
            optimizer.step()

            if epoch % 10 == 0:
                model.eval()
                with torch.no_grad():
                    val_logits = model(fold_data, val_edges)
                    val_probs  = torch.sigmoid(val_logits).cpu().numpy()
                    val_auc    = roc_auc_score(val_target, val_probs)
                    val_aupr   = average_precision_score(val_target, val_probs)

                monitor_score = 0.5 * (val_auc + val_aupr)
                best_monitor = 0.5 * (best_val_auc + best_val_aupr) if best_val_auc >= 0 else -1.0

                if monitor_score > best_monitor:
                    best_val_auc   = val_auc
                    best_val_aupr  = val_aupr
                    best_val_probs = val_probs.copy()
                    patience       = 0
                    best_epoch     = epoch
                else:
                    patience += 1

                if patience > 6:
                    break

        print(f"    Best Val AUC: {best_val_auc:.4f} (epoch {best_epoch})")
        run_auc_scores.append(best_val_auc)
        all_fold_probs.append(best_val_probs)
        all_fold_labels.append(val_target)

    all_run_auc_scores.append(run_auc_scores)
    run_mean = np.mean(run_auc_scores)
    run_std  = np.std(run_auc_scores)
    print(f"\n  Run {run_idx+1} AUC: {run_mean:.4f} ± {run_std:.4f}")

all_run_auc_scores = np.array(all_run_auc_scores)
run_means = all_run_auc_scores.mean(axis=1)
print(f"\n{'='*60}")
print(f"Per-run mean AUCs: {np.round(run_means, 4)}")
print(f"Overall ({n_runs} runs × {k_folds} folds): {run_means.mean():.4f} ± {run_means.std():.4f}")

save_dir = Path("results")
save_dir.mkdir(exist_ok=True)

np.save(save_dir / "HGT_fold_probs.npy",
        np.array(all_fold_probs, dtype=object), allow_pickle=True)
np.save(save_dir / "HGT_fold_labels.npy",
        np.array(all_fold_labels, dtype=object), allow_pickle=True)

all_probs  = np.concatenate(all_fold_probs)
all_labels = np.concatenate(all_fold_labels)
np.save(save_dir / "HGT_all_probs.npy",  all_probs)
np.save(save_dir / "HGT_all_labels.npy", all_labels)
np.save(save_dir / "HGT_run_auc_scores.npy", all_run_auc_scores)

cv_split_package = {
    "version": 1,
    "k_folds": int(k_folds),
    "n_runs": int(n_runs),
    "run_seeds": np.asarray(run_seeds, dtype=np.int64),
    "num_pos_edges": int(num_pos_edges),
    "num_tcr": int(num_tcr),
    "num_epitope": int(num_epitope),
    "runs": cv_split_runs,
}
cv_matrix_dir = Path("cv_matrix")
cv_matrix_dir.mkdir(exist_ok=True)
np.save(cv_matrix_dir / "HGT_cv_split_package.npy", cv_split_package, allow_pickle=True)

print(f"\nSaved to {save_dir}/:")
print(f"  HGT_fold_probs.npy   — {len(all_fold_probs)} folds ({n_runs} runs × {k_folds})")
print(f"  HGT_fold_labels.npy  — {len(all_fold_labels)} folds")
print(f"  HGT_all_probs.npy    — shape: {all_probs.shape}")
print(f"  HGT_all_labels.npy   — shape: {all_labels.shape}")
